# Production Planning & Supply Contract Optimization with Gurobi

**Linear Programming · Sensitivity Analysis · Fixed-Charge MILP · Scenario Optimization**

This portfolio notebook reformats a graduate Red Brand Canners optimization workshop into a recruiter-facing case study. The original work was completed as a **three-person graduate project**; this repository is a professional portfolio adaptation maintained by Franbon Ahmed Mohammed.

> The original copyrighted case handout is intentionally **not included**. The notebook uses only the mathematical formulation, derived results, and portfolio extensions needed to explain the optimization work.

## 1. Business problem

A food manufacturer must allocate two grades of tomatoes across three products—whole tomatoes, juice, and paste—while respecting demand, supply, and quality constraints.

The analysis answers six management questions:

1. What production mix maximizes contribution?
2. Should additional Grade A tomatoes be purchased?
3. Which product should receive advertising support?
4. Is additional Grade B supply worth buying?
5. Which production lines should be opened when each line has a fixed setup cost?
6. How much raw material should be contracted when crop quality is uncertain?

## 2. Base linear program

Decision variables represent thousands of pounds of Grade A and Grade B tomatoes allocated to whole tomatoes (`Aw`, `Bw`), juice (`Aj`, `Bj`), and paste (`Ap`, `Bp`).

The source model maximizes contribution margin subject to product-demand limits, Grade A / B supply, and minimum quality requirements.

In [ ]:
import gurobipy as gp
from gurobipy import GRB

m = gp.Model("RBC_base")
Aw = m.addVar(lb=0, name="Aw"); Bw = m.addVar(lb=0, name="Bw")
Aj = m.addVar(lb=0, name="Aj"); Bj = m.addVar(lb=0, name="Bj")
Ap = m.addVar(lb=0, name="Ap"); Bp = m.addVar(lb=0, name="Bp")

m.setObjective(246.67*(Aw+Bw) + 198*(Aj+Bj) + 222*(Ap+Bp), GRB.MAXIMIZE)
m.addConstr(Aw + Bw <= 14400, name="dem_whole")
m.addConstr(Aj + Bj <= 1000, name="dem_juice")
m.addConstr(Ap + Bp <= 2000, name="dem_paste")
m.addConstr(Aw + Aj + Ap <= 600, name="sup_A")
m.addConstr(Bw + Bj + Bp <= 2400, name="sup_B")
m.addConstr(Aw >= 3*Bw, name="qual_whole")
m.addConstr(Bj <= 3*Aj, name="qual_juice")
m.optimize()

### Source result

The saved report gives an optimal contribution of **$676,069**. The production allocation is 700 thousand lb to whole tomatoes, 300 thousand lb to juice, and 2,000 thousand lb to paste. Paste demand and both raw-material supply constraints are binding.

![Base production mix](../images/product_mix.png)

The sensitivity analysis shows why the solution looks this way: the marginal value of Grade A supply is higher than Grade B, while additional whole and juice demand has no value at the base optimum.

![Shadow prices](../images/sensitivity_shadow_prices.png)

## 3. Additional Grade A decision

The source model allows up to **80,000 lb** of additional Grade A tomatoes at **25.5 cents/lb**. The saved solution buys the full amount and raises the objective to **$677,349.40**.

The extra-A capacity shadow price implies a break-even purchase price of approximately **27.10 cents/lb**. The source analysis also identifies multiple optimal allocations of the purchased Grade A, so the management recommendation should not imply a unique physical allocation.

## 4. Advertising and marginal resource value

Increasing demand by 5,000 cases produces no objective gain for whole tomatoes or juice in the saved source analysis, but increases profit by **$6,041.88** for paste.

**Decision:** direct the campaign to paste and pay no more than approximately **$6,041.88** for that demand increase.

For extra Grade B supply, the shadow price is approximately **17.37 cents/lb**, below the offered **18 cents/lb**.

**Decision:** do not buy the additional Grade B at that price.

## 5. Fixed setup costs: original enumeration

The original assignment evaluated every non-empty combination of production lines and subtracted **$50,000 per opened line**. The best source result is **Juice + Paste**, with net profit of **$542,000**.

![Setup comparison](../images/setup_cost_comparison.png)

## 6. Portfolio extension: formulate Part 5 as a MILP

Enumeration is valid for three lines, but it does not scale well. A more general fixed-charge formulation introduces binary setup variables:

- `y_whole ∈ {0,1}`
- `y_juice ∈ {0,1}`
- `y_paste ∈ {0,1}`

Production is linked to the corresponding binary variable, and the $50,000 setup charge is placed directly in the objective. This is a **portfolio extension added after the original assignment**.

In [ ]:
m = gp.Model("RBC_fixed_charge_MILP")
x = {n: m.addVar(lb=0, name=n) for n in ["Aw","Bw","Aj","Bj","Ap","Bp"]}
y = {p: m.addVar(vtype=GRB.BINARY, name=f"open_{p}") for p in ["whole","juice","paste"]}
setup = 50000

m.setObjective(
    246.67*(x["Aw"]+x["Bw"])
    + 198*(x["Aj"]+x["Bj"])
    + 222*(x["Ap"]+x["Bp"])
    - setup*(y["whole"]+y["juice"]+y["paste"]),
    GRB.MAXIMIZE
)

m.addConstr(x["Aw"]+x["Bw"] <= 14400*y["whole"])
m.addConstr(x["Aj"]+x["Bj"] <= 1000*y["juice"])
m.addConstr(x["Ap"]+x["Bp"] <= 2000*y["paste"])
m.addConstr(x["Aw"]+x["Aj"]+x["Ap"] <= 600)
m.addConstr(x["Bw"]+x["Bj"]+x["Bp"] <= 2400)
m.addConstr(x["Aw"] >= 3*x["Bw"])
m.addConstr(x["Bj"] <= 3*x["Aj"])
m.optimize()

**Validation target:** this MILP is mathematically equivalent to the original line-opening decision and should reproduce the source recommendation to open **Juice + Paste** with net profit **$542,000**, subject to using the same coefficients and Gurobi environment.

## 7. Contracting under uncertain quality

The source analysis considers sunny, normal, and poor crop-quality scenarios with probabilities **25% / 50% / 25%**. If management is restricted to the scenario-specific candidate orders, the poor-year order of about **2.73M lb** has the highest expected profit among those three candidates.

The source project then searches feasible contract quantities from 0 to 13M lb in **50,000-lb increments** and finds a better tested policy: **3.65M lb**, with expected profit **$125,716.47**.

![Expected profit curve](../images/contract_expected_profit.png)

The curve above is a **portfolio visualization extension** generated from the same LP coefficients and scenario probabilities. It reproduces the source report's best tested grid point at approximately 3.65M lb.

![Scenario profits](../images/scenario_profit_comparison.png)

## 8. Management recommendations

| Decision | Recommendation |
|---|---|
| Base production | Produce paste at capacity plus limited whole and juice |
| Extra Grade A | Buy the full 80,000 lb at 25.5 cents/lb |
| Advertising | Target paste; value up to about $6,041.88 |
| Extra Grade B | Reject at 18 cents/lb |
| Setup costs | Open Juice + Paste |
| Contract quantity | About 3.65M lb on the tested grid |

The broader lesson is that optimization adds value not only by producing a single solution, but by quantifying **marginal resource values, break-even prices, fixed-charge tradeoffs, and risk under uncertainty**.

## 9. Project origin and responsible sharing

This project originated as a **three-person graduate optimization workshop**. This repository is a portfolio adaptation maintained by Franbon Ahmed Mohammed.

The original workshop handout is not redistributed because the course document states that it may not be reproduced without permission. The repository therefore presents only the mathematical formulation, derived analysis, and original portfolio extensions.